# 04 SQL Analysis

This notebook uses SQLite to query the cleaned SkillMap job-market database.

The goal is to practice SQL analysis on the UAE and Canada job-posting dataset by answering questions about countries, skills, salary disclosure, seniority, work mode, and entry-level experience expectations.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

In [2]:
DATABASE_PATH = Path("../data/processed/skillmap.db")

connection = sqlite3.connect(DATABASE_PATH)

In [3]:
def run_query(query):
    return pd.read_sql_query(query, connection)

In [4]:
run_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table';
""")

,name
0,jobs
1,job_skills
2,sqlite_sequence


In [5]:
run_query("""
SELECT COUNT(*) AS total_jobs
FROM jobs;
""")

,total_jobs
0,20


In [6]:
run_query("""
SELECT COUNT(*) AS total_skill_rows
FROM job_skills;
""")

,total_skill_rows
0,133


In [7]:
run_query("""
SELECT
    country,
    COUNT(*) AS job_count
FROM jobs
GROUP BY country
ORDER BY job_count DESC;
""")

,country,job_count
0,Canada,10
1,UAE,10


In [8]:
run_query("""
SELECT
    country,
    COUNT(*) AS total_jobs,
    SUM(CASE WHEN has_salary = 1 THEN 1 ELSE 0 END) AS salary_disclosed_jobs,
    ROUND(
        SUM(CASE WHEN has_salary = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS salary_disclosure_percentage
FROM jobs
GROUP BY country
ORDER BY country;
""")

,country,total_jobs,salary_disclosed_jobs,salary_disclosure_percentage
0,Canada,10,10,100.0
1,UAE,10,1,10.0


### Interpretation

This query compares salary transparency across the UAE and Canada samples. It counts how many postings disclosed salary information and calculates the disclosure rate within each country.

The result should be interpreted carefully because the dataset contains only 20 manually collected postings. It shows salary disclosure patterns in this sample, not a definitive conclusion about the full UAE or Canadian job market.

Salary amounts should also not be compared directly yet because some salaries are hourly, some are annual, and some postings do not disclose salary at all.

In [15]:
run_query("""
SELECT
    skill,
    COUNT(DISTINCT job_id) AS job_count,
    ROUND(
        COUNT(DISTINCT job_id) * 100.0 / (SELECT COUNT(*) FROM jobs),
        1
    ) AS percentage_of_all_jobs
FROM job_skills
GROUP BY skill
ORDER BY job_count DESC, skill ASC;
""")

,skill,job_count,percentage_of_all_jobs
0,Reporting,16,80.0
1,Stakeholder Communication,16,80.0
2,SQL,10,50.0
3,Technical Documentation,10,50.0
4,Dashboarding,9,45.0
5,Data Visualization,9,45.0
6,Excel,8,40.0
7,Power BI,8,40.0
8,Python,8,40.0
9,Presentation Skills,7,35.0


### Interpretation

This query shows the most frequently mentioned skills across the full 20-posting dataset, regardless of country. It provides a quick overall view of which skills appear most often before comparing UAE and Canada separately.

The result is useful as a baseline, but the country-level skill analysis is more informative because the project’s main goal is to compare UAE and Canadian job-market patterns.

In [10]:
run_query("""
SELECT
    js.country,
    js.skill,
    COUNT(DISTINCT js.job_id) AS job_count,
    ROUND(
        COUNT(DISTINCT js.job_id) * 100.0 /
        (
            SELECT COUNT(*)
            FROM jobs j
            WHERE j.country = js.country
        ),
        1
    ) AS percentage_of_country_jobs
FROM job_skills js
GROUP BY js.country, js.skill
ORDER BY js.country, job_count DESC, js.skill ASC;
""")

,country,skill,job_count,percentage_of_country_jobs
0,Canada,Reporting,8,80.0
1,Canada,Stakeholder Communication,8,80.0
2,Canada,Technical Documentation,8,80.0
3,Canada,Python,6,60.0
4,Canada,SQL,6,60.0
5,Canada,Data Visualization,4,40.0
6,Canada,Power BI,4,40.0
7,Canada,Quality Assurance,4,40.0
8,Canada,Dashboarding,3,30.0
9,Canada,Data Cleaning,3,30.0


### Interpretation

This query shows which skills appear most frequently in each country sample. The percentage column is more useful than raw counts because each country has the same number of collected postings.

In this sample, the results show that analytics roles combine technical skills with business-facing skills. Skills such as reporting, stakeholder communication, documentation, dashboarding, SQL, Python, Excel, and Power BI are especially important because they appear across multiple postings.

Because this is a small sample, these findings should be treated as directional indicators rather than complete market-wide rankings.

In [11]:
run_query("""
SELECT
    job_id,
    country,
    job_title,
    company,
    seniority,
    years_experience_min,
    years_experience_max,
    notes
FROM jobs
WHERE seniority IN ('Entry-level', 'Junior', 'Associate')
  AND years_experience_min >= 2
ORDER BY country, years_experience_min DESC;
""")

,job_id,country,job_title,company,seniority,years_experience_min,years_experience_max,notes
0,CAN_006,Canada,Junior Data Analyst,Procom,Junior,2.0,3.0,Salary listed hourly; 6-month contract; hybrid...
1,UAE_004,UAE,Associate Data Analyst- UAE Nationals only,Delivery Hero SE,Associate,2.0,3.0,UAE nationals only; early-career wording but r...
2,UAE_005,UAE,Associate Data Analyst,Bayut | dubizzle,Associate,2.0,3.0,NaN
3,UAE_010,UAE,Data Analyst – Performance Marketing,Cheil Middle East & Africa,Entry-level,2.0,3.0,LinkedIn labels entry-level; posting asks for ...


### Interpretation

This query identifies early-career roles that still require at least 2 years of experience. This is important because job titles such as junior, associate, or entry-level do not always mean the role is truly beginner-friendly.

For job-search strategy, these roles may still be worth applying to if the candidate can show strong project evidence, internship experience, technical skills, and relevant transferable experience.

This finding should not be overgeneralized. It shows that some roles in this sample ask for prior experience, not that all entry-level roles require experience.

In [12]:
run_query("""
SELECT DISTINCT
    j.job_id,
    j.country,
    j.city,
    j.job_title,
    j.company,
    j.seniority
FROM jobs j
JOIN job_skills sql_skill
    ON j.job_id = sql_skill.job_id
JOIN job_skills powerbi_skill
    ON j.job_id = powerbi_skill.job_id
WHERE sql_skill.skill = 'SQL'
  AND powerbi_skill.skill = 'Power BI'
ORDER BY j.country, j.job_id;
""")

,job_id,country,city,job_title,company,seniority
0,CAN_001,Canada,Toronto,"Data and Business Intelligence Analyst, Intern...",Hitachi Rail Canada Inc.,Internship
1,CAN_002,Canada,Toronto,Business Analyst (Engineering) Intern (Fall 20...,Hitachi Rail Canada Inc.,Internship
2,UAE_004,UAE,Dubai,Associate Data Analyst- UAE Nationals only,Delivery Hero SE,Associate
3,UAE_009,UAE,Dubai,Jr. AI Business Analyst,Simply Solved Accounting & Bookkeeping LLC,Junior


### Interpretation

This query finds postings that mention both SQL and Power BI. This combination is important for data analyst and BI analyst roles because SQL supports data extraction and querying, while Power BI supports dashboarding and business reporting.

The query also demonstrates a more advanced SQL pattern by joining the `job_skills` table twice: once to check for SQL and once to check for Power BI.

For portfolio development, this supports prioritizing SQL and Power BI together rather than treating them as separate skills.

In [ ]:
run_query("""
SELECT DISTINCT
    j.job_id,
    j.country,
    j.job_title,
    j.company,
    j.seniority,
    js.skill
FROM jobs j
JOIN job_skills js
    ON j.job_id = js.job_id
WHERE js.skill IN (
    'Quality Assurance',
    'Manual Testing',
    'Bug Reporting',
    'Technical Documentation'
)
ORDER BY j.country, j.job_id, js.skill;
""")


,job_id,country,job_title,company,seniority,skill
0,CAN_001,Canada,"Data and Business Intelligence Analyst, Intern...",Hitachi Rail Canada Inc.,Internship,Quality Assurance
1,CAN_001,Canada,"Data and Business Intelligence Analyst, Intern...",Hitachi Rail Canada Inc.,Internship,Technical Documentation
2,CAN_003,Canada,data analyst - informatics and systems,Snappay Inc.,Junior,Technical Documentation
3,CAN_004,Canada,data quality analyst,Planta Greenhouses,Junior,Quality Assurance
4,CAN_004,Canada,data quality analyst,Planta Greenhouses,Junior,Technical Documentation
5,CAN_005,Canada,"business analyst, informatics",IMDS Software Inc.,Junior,Manual Testing
6,CAN_005,Canada,"business analyst, informatics",IMDS Software Inc.,Junior,Quality Assurance
7,CAN_005,Canada,"business analyst, informatics",IMDS Software Inc.,Junior,Technical Documentation
8,CAN_006,Canada,Junior Data Analyst,Procom,Junior,Technical Documentation
9,CAN_008,Canada,September 2026 IT Analytics Coop Student,ATCO Ltd. - Common Groups,Co-op,Manual Testing


### Interpretation

This query identifies roles that overlap with QA, testing, bug reporting, or technical documentation. This is useful because these skills connect directly to existing product, testing, and documentation experience.

The result supports a stronger career positioning: the transition into analytics does not rely only on learning new technical tools. Existing experience in testing, documentation, user acceptance testing, and product support can also be relevant for analyst, product analyst, BI, and QA-data hybrid roles.

This is especially useful for explaining how previous UI/UX and product-team experience connects to data-focused roles.

## Phase 5 SQL Summary

This phase converted the cleaned job-posting CSV files into a SQLite database with two related tables:

- `jobs`: one row per job posting
- `job_skills`: one row per job-skill relationship

The SQL analysis covered:

- database validation
- job counts by country
- seniority distribution
- work mode distribution
- salary disclosure
- experience expectations
- top skills overall
- top skills by country
- technical tool demand
- roles requiring both SQL and Power BI
- roles overlapping with QA, testing, and documentation skills

The strongest SQL skills demonstrated in this phase include:

- filtering with `WHERE`
- grouping with `GROUP BY`
- conditional aggregation with `CASE WHEN`
- percentage calculations
- `COUNT(DISTINCT ...)`
- joining related tables
- joining the same table multiple times for multi-skill filtering

This database structure makes the project easier to query, supports more reliable analysis than working from one flat CSV alone, and prepares the cleaned data for Power BI dashboard development.